In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import Dataset, load_dataset
from IPython.display import display
import datetime
import numpy
import os
import pandas as pd
import random
import torch
import torch.nn.functional as F
import json
from sentence_transformers import (
    InputExample,
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
    losses
)
from sentence_transformers.sentence_transformer.evaluation import BinaryClassificationEvaluator, RerankingEvaluator
from transformers import (
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoTokenizer
)

/tmp/ipykernel_1595/3865950663.py:11: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


## **Loading the dataset and initialize features**

In [ ]:
# Dataset loading
def load_data():
    ds = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")

    ds_train = ds["train"]

    # Split the train set into train and dev sets
    train_dev_split = ds["train"].train_test_split(test_size=0.1, seed=42)

    ds_train = train_dev_split["train"]
    ds_dev = train_dev_split["test"]

    return {
        "train": ds_train,
        "dev": ds_dev,
        "test": ds["test"],
        "blind": ds["blind"]
    }

In [ ]:
ds = load_data()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating blind split:   0%|          | 0/1322 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

### Baseline Implementation
Here we define our custom functions for the baseline implementation for the following operations: mean pooling, encoding, embedding.

In [ ]:
# Mean pooling function
def mean_pooling(model_output, attention_mask):
  token_embeddings = model_output[0]
  input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
  sum = torch.sum(token_embeddings * input_mask_expanded, 1)
  counts = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
  return sum / counts

In [ ]:
# Optimized encoding function with batching support
def encode_texts(texts, tokenizer, model, batch_size=16):
  all_embeddings = []
  device = next(model.parameters()).device
  # device = torch.device('mps') if torch.backend.mps.is_available() else torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu') # Ensures to use GPU if it is possible
  # model.to(device)

  for i in range(0, len(texts), batch_size): # Break data into small chunks in order to avoid problems of memory
    batch_texts = texts[i:i + batch_size]
    inputs = tokenizer(batch_texts,
                       padding=True, # Used to have sentences with same length
                       truncation=True, # Cut too long sentences
                       return_tensors='pt').to(device)

    with torch.no_grad(): # Disables 'torch' gradients in order to save time (and performance)
      outputs = model(**inputs)
      embeddings = mean_pooling(outputs, inputs['attention_mask'])
      normalized_embeddings = torch.nn.functional.normalize(embeddings , p=2, dim=1).cpu()  # Normalize and move back to CPU to save GPU memory
      all_embeddings.append(normalized_embeddings)

  return torch.cat(all_embeddings, dim=0)

In [ ]:
# Optimized embedding function using batching for queries and candidates
def embedding(queries, candidate_chunks_list, tokenizer, model):
  device = next(model.parameters()).device # added line for GPU
  print("Encoding all queries...")
  # Use the model's native encode if it's a SentenceTransformer, otherwise use our helper
  if isinstance(model, SentenceTransformer):
    query_embeddings = model.encode(queries, batch_size=64, device=device, convert_to_tensor=True, normalize_embeddings=True).cpu()
  else:
    query_embeddings = encode_texts(queries, tokenizer, model, batch_size=64)

  # Flatten the candidates for better GPU performance
  flat_candidates = [chunk for sublist in candidate_chunks_list for chunk in sublist]

  print("Encoding all candidate chunks...")
  if isinstance(model, SentenceTransformer):
    flat_candidate_embeddings = model.encode(flat_candidates, batch_size=64, device=device, convert_to_tensor=True, normalize_embeddings=True).cpu()
  else:
    flat_candidate_embeddings = encode_texts(flat_candidates, tokenizer, model, batch_size=64)

  candidate_embeddings = []
  start_idx = 0

  # Regroup the flattened candidates
  print(f"Regrouping candidate chunks for {len(queries)} queries...")
  for chunks in candidate_chunks_list:
    num_chunks = len(chunks)
    group_emb = flat_candidate_embeddings[start_idx : start_idx + num_chunks]
    candidate_embeddings.append(group_emb)
    start_idx += num_chunks

  return query_embeddings, candidate_embeddings

### **Setting up the two baseline models: DistilBERT and all-MiniLM-L6-v2**
Here, we initialise two models to establish our points of comparison:
- DistilBERT Base: The model requested for fine-tuning.
- All-MiniLM-L6-v2: A robust baseline already optimised for semantic similarity.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# Load pre-trained models and tokenizers
distilbert = 'distilbert/distilbert-base-uncased'
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert)
distilbert_model = SentenceTransformer(distilbert, device=device)

all_mini = 'sentence-transformers/all-MiniLM-L6-v2'
all_mini_tokenizer = AutoTokenizer.from_pretrained(all_mini)
all_mini_model = SentenceTransformer(all_mini, device=device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

We now generate the embeddings for queries and candidates over the test set.

In [ ]:
# Generate embeddings for test and blind sets
distilbert_query_embeddings, distilbert_candidate_embeddings = embedding(ds["test"]["query"], ds["test"]["candidate_chunks"], distilbert_tokenizer, distilbert_model)
all_mini_query_embeddings, all_mini_candidate_embeddings = embedding(ds["test"]["query"], ds["test"]["candidate_chunks"], all_mini_tokenizer, all_mini_model)

print(f"Generated {len(distilbert_query_embeddings)} query embeddings.")
print(f"Generated {len(distilbert_candidate_embeddings)} lists of candidate embeddings.")


Encoding all queries...
Encoding all candidate chunks...
Regrouping candidate chunks for 2000 queries...
Encoding all queries...
Encoding all candidate chunks...
Regrouping candidate chunks for 2000 queries...
Generated 2000 query embeddings.
Generated 2000 lists of candidate embeddings.


### Hit@k metrics
Here we calculate Hit@k metrics (k=1, 3, 5) on the test split before any training. This allows us to quantify baseline performance and measure the future impact of fine-tuning.

Both cosine similarity and euclidean distance similarity are defined for further evaluation purposes.

In [ ]:
def hit_at_k(query_embeddings, candidate_embeddings, answer_pos, distance_metric, ks=[1, 3, 5]):
    hit_counts = {k: 0 for k in ks}
    N = len(query_embeddings)

    for i, query in enumerate(query_embeddings):
      # candidate_embeddings[i] is a tensor of all candidate vectors for query i
      candidates = candidate_embeddings[i]

      if distance_metric == 'cosine':
          scores = torch.nn.functional.cosine_similarity(query.unsqueeze(0), candidates, dim=1)
          descending = True
      elif distance_metric == 'euclidean':
          scores = torch.cdist(query.unsqueeze(0), candidates, p=2).squeeze(0)
          descending = False

      # Get indices of candidates ranked by distance
      sorted_indices = torch.argsort(scores, descending=descending)

      # .item() gets the integer value from the single-element tensor
      rank = (sorted_indices == answer_pos[i]).nonzero(as_tuple=True)[0].item()

      # If rank is 0, it's the top 1. If rank < 3, it's in top 3, etc.
      for k in ks:
          if rank < k:
              hit_counts[k] += 1

    # Average the hits over all queries
    return {f"Hit@{k}": hit_counts[k] / N for k in ks}

In [ ]:
# Ensure we use the correct answer positions for the test set
distilbert_results = hit_at_k(distilbert_query_embeddings, distilbert_candidate_embeddings, ds["test"]["answer_pos"], distance_metric='cosine')
print(f"Hit@K metrics results for the {distilbert}: {distilbert_results}")

all_mini_results = hit_at_k(all_mini_query_embeddings, all_mini_candidate_embeddings, ds["test"]["answer_pos"], distance_metric='cosine')
all_mini_results = {k: f"{v:.4f}" for k, v in all_mini_results.items()}
print(f"Hit@K metrics results for the {all_mini}: {all_mini_results}")


Hit@K metrics results for the distilbert/distilbert-base-uncased: {'Hit@1': 0.1765, 'Hit@3': 0.4435, 'Hit@5': 0.6045}
Hit@K metrics results for the sentence-transformers/all-MiniLM-L6-v2: {'Hit@1': '0.4480', 'Hit@3': '0.7170', 'Hit@5': '0.8080'}


In [ ]:
data = {
    "Model": [distilbert, all_mini],
    "Hit@1 ": [distilbert_results['Hit@1'], all_mini_results['Hit@1']],
    "Hit@3": [distilbert_results['Hit@3'], all_mini_results['Hit@3']],
    "Hit@5": [distilbert_results['Hit@5'], all_mini_results['Hit@5']]
}
df_results = pd.DataFrame(data)
display(df_results.style.format(precision=4).set_caption("Baselines Comparison").hide(axis='index'))

Model,Hit@1,Hit@3,Hit@5
distilbert/distilbert-base-uncased,0.1765,0.4435,0.6045
sentence-transformers/all-MiniLM-L6-v2,0.4480,0.7170,0.8080


# Fine-Tuning Strategies

## Sentence Pairs
In this section, we provide a first fine-tuning strategy: sentence pairs. The idea is to try a very simple strategy, in order to compare how fine-tuning is able to improve distilbert model's performance over the test set.

We first define our function for creating two pairs for each query: one is the positive pair (query, answer), while the second one is the negative pair (query, candidate), where the candidate is chosen at random from the candidate chunks.

In [ ]:
def create_sentence_pairs(dataset, query_embeddings, candidates_embeddings, seed=42):
    random.seed(seed)
    paired_dataset = [] # this will be our final dataset in a List object
    pair = {} # we define a Dict structure for each pair
    ds_length = len(dataset)

    # Iterate over rows of the dataset
    for row_idx, ds_line in enumerate(dataset):
        query = ds_line["query"]
        candidates = ds_line["candidate_chunks"]
        answer_pos = ds_line["answer_pos"]

        positive = candidates[answer_pos] # The correct answer is the positive example for the query

        # Positive pair binding
        pair = {
            "sentence1": query,
            "sentence2": positive,
            "label": 1.0,
        }
        paired_dataset.append(pair)

        # Remove the current line index to avoid picking the same query as a negative example
        line_idx = list(range(ds_length))
        line_idx.remove(row_idx)

        # Pick 2 random queries from the dataset to create negative pairs with the current query
        for i in range(2):
            random_idx = random.choice(line_idx)
            random_query = dataset[random_idx]["query"]

            # Negative pair binding
            pair = {
                "sentence1": query,
                "sentence2": random_query,
                "label": 0.0,
            }
            paired_dataset.append(pair)

            line_idx.remove(random_idx)

        # Compute cosine similarities and sort the negatives by similarity to the query
        similarities = F.cosine_similarity(candidates_embeddings[row_idx], query_embeddings[row_idx].unsqueeze(0)).squeeze()
        sorted_indices = torch.argsort(similarities, descending=True)
        negatives = [candidates[idx] for idx in sorted_indices if idx != answer_pos] # Exclude the positive example from the negatives

        # Pick the top 3 most similar negative candidates and create pairs
        for negative in negatives[:3]:
            # Negative pair binding
            pair = {
                "sentence1": query,
                "sentence2": negative,
                "label": 0.0,
            }
            paired_dataset.append(pair)

    return Dataset.from_list(paired_dataset)

## Multiple Negatives Ranking
In this section, we try a different fine-tuning strategy for our distilbert model. The intuition is that, since every query has a greate set of negative candidates and just a single positive one, using a loss that helps the model distantiate a tuple of n negative candidates from the positive one is a more complex, but more efficient approach with respect to the Sentence Pairs strategy.

We define our functions in order to split the dataset into batches in the requested input format: (anchor, positive, negative_1, ..., negative_n). Of course, 'anchor' is, the specific query, the 'positive' is the answer, 'negative_1' to 'negative_n' are the negative candidates,

In [ ]:
def create_batches(dataset, query_embeddings, candidates_embeddings):
  batch = {
      "anchor": [],
      "positive": [],
      "negative_1": [],
      "negative_2": [],
      "negative_3": []
  }

  # Pick 3 negative candidates at random for each query (every query has at least 5 candidates, answer included)
  for row_idx, ds_line in enumerate(dataset):
    query = ds_line["query"]
    candidates = ds_line["candidate_chunks"]
    n_candidates = ds_line["n_candidates"]
    answer = ds_line["answer"]
    answer_pos = ds_line["answer_pos"]

    # Add the query and the positive example to the batch
    batch["anchor"].append(query)
    batch["positive"].append(answer)

    # Remove the positive example from the candidates to avoid picking it as a negative example
    indices = list(range(n_candidates))
    indices.remove(answer_pos)

    # Compute cosine similarities and sort the negatives by similarity to the query
    similarities = F.cosine_similarity(candidates_embeddings[row_idx], query_embeddings[row_idx].unsqueeze(0)).squeeze()
    sorted_indices = torch.argsort(similarities, descending=True)
    sorted_indices = [idx.item() for idx in sorted_indices if idx.item() != answer_pos]

    # Pick the top 3 most similar negative candidates and add them to the batch
    for i in range(3):
      batch[f"negative_{i+1}"].append(candidates[sorted_indices[i]])

  return Dataset.from_dict(batch)

In [ ]:
def create_samples(mnr_dev_dataset): # Creates samples to be used for evaluation with the RerankingEvaluator
    samples = []

    # Create a sample for each query in the dev set
    for ds_line in mnr_dev_dataset:
        negatives = []
        for i in range(1, 4):
            negatives.append(ds_line[f"negative_{i}"])
            samples.append({
                "query": ds_line["anchor"],
                "positive": [ds_line["positive"]],
                "negative": negatives
            })
    return samples

## Training and Evaluation
We are now ready to define all the needed functions for our training and evaluation phases over our models and strategies.

In [ ]:
def train_args(model, strategy, learning_rate=2e-5, epochs=3, warmup_steps=0.1) -> SentenceTransformerTrainingArguments:
  return SentenceTransformerTrainingArguments(
    output_dir="models/" + model + "-" + strategy + "-" + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
    learning_rate=learning_rate,
    num_train_epochs=epochs,
    warmup_steps=warmup_steps,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4, # Compensates for the small batch size by accumulating gradients over multiple steps
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    run_name=model
  )


In [ ]:
def sentence_transformer_trainer(model, args, train_set, dev_set, loss, evaluator) -> SentenceTransformerTrainer:
  return SentenceTransformerTrainer(
      model=model,
      args=args,
      train_dataset=train_set,
      eval_dataset=dev_set,
      loss=loss,
      evaluator=evaluator
  )

In [ ]:
models = {
    "distilbert_pairs": {
        "model_name": "distilbert/distilbert-base-uncased",
        "strategy": "pairs",
        "loss": losses.ContrastiveLoss,
        "strategy": "pairs",
    },
    "all-mini_pairs": {
        "model_name": "sentence-transformers/all-miniLM-L6-v2",
        "strategy": "pairs",
        "loss": losses.ContrastiveLoss,
        "strategy": "pairs",
    },
    "f2llm-pairs": {
        "model_name": "codefuse-ai/F2LLM-v2-80M",
        "strategy": "pairs",
        "loss": losses.ContrastiveLoss,
        "strategy": "pairs",
    },
    "distilbert_mnr": {
        "model_name": "distilbert/distilbert-base-uncased",
        "strategy": "mnr",
        "loss": losses.MultipleNegativesRankingLoss,
        "strategy": "mnr",
    },
    "all-mini_mnr": {
        "model_name": "sentence-transformers/all-miniLM-L6-v2",
        "strategy": "mnr",
        "loss": losses.MultipleNegativesRankingLoss,
        "strategy": "mnr",
    },
    "f2llm-mnr": {
        "model_name": "codefuse-ai/F2LLM-v2-80M",
        "strategy": "mnr",
        "loss": losses.MultipleNegativesRankingLoss,
        "strategy": "mnr",
    }
}

We define the embedding process for queries and candidates, for both train and dev sets.

In [ ]:
embeddings_file = "embeddings.pt"

if os.path.exists(embeddings_file):
    saved_embeddings = torch.load(embeddings_file, weights_only=True)

    train_query_embeddings = saved_embeddings["train_query"]
    train_candidates_embeddings = saved_embeddings["train_candidates"]
    dev_query_embeddings = saved_embeddings["dev_query"]
    dev_candidates_embeddings = saved_embeddings["dev_candidates"]
else:
    # Embedding model and tokenizer loading
    embedding_model = SentenceTransformer("avsolatorio/GIST-small-Embedding-v0", device=device)
    embedding_tokenizer = AutoTokenizer.from_pretrained("avsolatorio/GIST-small-Embedding-v0")

    # Embedding computation for both queries and candidates, for both train an dev sets.
    train_query_embeddings, train_candidates_embeddings = embedding(ds["train"]["query"], ds["train"]["candidate_chunks"], embedding_tokenizer, embedding_model)
    dev_query_embeddings, dev_candidates_embeddings = embedding(ds["dev"]["query"], ds["dev"]["candidate_chunks"], embedding_tokenizer, embedding_model)

    torch.save({
        "train_query": train_query_embeddings,
        "train_candidates": train_candidates_embeddings,
        "dev_query": dev_query_embeddings,
        "dev_candidates": dev_candidates_embeddings
    }, embeddings_file)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding all queries...
Encoding all candidate chunks...
Regrouping candidate chunks for 7200 queries...
Encoding all queries...
Encoding all candidate chunks...
Regrouping candidate chunks for 800 queries...


Now we are ready to start the training process. A for cycle is defined in order to train different models with the same parameters.

In [ ]:
# For each model in the dictionary, we create the appropriate training and evaluation structure
for _, model_item in models.items():

  model = SentenceTransformer(model_item["model_name"]) # Load the pre-trained model

  print(f"Training {model_item['model_name']} with {model_item['strategy']} strategy...")

  if model_item["strategy"] == "pairs": # We create sentence pairs for the contrastive loss strategy, and a BinaryClassificationEvaluator for evaluation
      ds_train = create_sentence_pairs(ds["train"], train_query_embeddings, train_candidates_embeddings)
      ds_dev = create_sentence_pairs(ds["dev"], dev_query_embeddings, dev_candidates_embeddings)
      dev_evaluator = BinaryClassificationEvaluator(
          sentences1=ds_dev["sentence1"],
          sentences2=ds_dev["sentence2"],
          labels=ds_dev["label"],
          name=model_item["model_name"] + "-pairs-dev",
          write_csv=False
      )

  elif model_item["strategy"] == "mnr": # We create batches with multiple negatives for the Multiple Negatives Ranking Loss strategy, and a RerankingEvaluator for evaluation
      ds_train = create_batches(ds["train"], train_query_embeddings, train_candidates_embeddings)
      ds_dev = create_batches(ds["dev"], dev_query_embeddings, dev_candidates_embeddings)
      dev_evaluator = RerankingEvaluator(
          samples=create_samples(ds_dev),
          name=model_item["model_name"] + "-mnr-dev",
          write_csv=False
      )

  loss = model_item["loss"](model) # Inizialiting the loss function with the model

  args = train_args(model_item["model_name"], model_item["strategy"], learning_rate=4e-5, warmup_steps=0.15) # Getting the training arguments for the model and strategy

  trainer = sentence_transformer_trainer(model, args, ds_train, ds_dev, loss, dev_evaluator) # Creating the trainer for the model and strategy

  # Training and saving the model
  trainer.train()
  model.save_pretrained(args.output_dir)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training distilbert/distilbert-base-uncased with pairs strategy...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Distilbert/distilbert-base-uncased-pairs-dev Cosine Accuracy,Distilbert/distilbert-base-uncased-pairs-dev Cosine Accuracy Threshold,Distilbert/distilbert-base-uncased-pairs-dev Cosine F1,Distilbert/distilbert-base-uncased-pairs-dev Cosine F1 Threshold,Distilbert/distilbert-base-uncased-pairs-dev Cosine Precision,Distilbert/distilbert-base-uncased-pairs-dev Cosine Recall,Distilbert/distilbert-base-uncased-pairs-dev Cosine Ap,Distilbert/distilbert-base-uncased-pairs-dev Cosine Mcc
1,0.014762,0.011738,0.872708,0.752421,0.598778,0.651810,0.505155,0.735000,0.629532,0.513895
2,0.010491,0.011302,0.878125,0.742179,0.605678,0.671640,0.522686,0.720000,0.658562,0.521491
3,0.006625,0.011484,0.876250,0.744664,0.607440,0.667012,0.546454,0.683750,0.653914,0.523105


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Training sentence-transformers/all-miniLM-L6-v2 with pairs strategy...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss
